# P2 MobileViTv2 ImageNet + LI architecture control

**Scientific question:** What is the architecture contribution relative to P1-noSSL when initialization family, LI fusion, targets, loss, split, seed, and supervised protocol are fixed?  
**Configuration:** `config/experiments/P2_mobilevitv2_imagenet_li_bestreg.yaml`  
**Dataset:** 1,927 clean unique labeled images.  
**Split:** 1,407 train / 289 validation / 231 sealed test; SHA256 `8927b822...3eae2f`.  
**Checkpoint/initialization:** `timm/mobilevitv2_050.cvnets_in1k` ImageNet weights; no VICReg.  
**Expected outputs:** external P2 best/epoch-60 checkpoints, history, validation artifacts, metadata, then the four-model frozen registry.

Run after notebooks 01–03 have completed. This notebook trains for all 60 epochs and never creates a test dataset or loader.


## 1. Deterministic environment setup

Set the CUDA deterministic workspace before importing PyTorch, then load only train/validation workflow functions.


In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
from pathlib import Path
import sys

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
if not (REPO / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from inside the SoilNet repository")
sys.path.insert(0, str(REPO / "src"))

import json
import torch
from soilnet.final_sequence import verify_p2_fairness, write_frozen_model_registry
from soilnet.training import run_one_batch_preflight, train_experiment
from soilnet.utils import load_experiment_context, print_environment

CONFIG_PATH = REPO / "config/experiments/P2_mobilevitv2_imagenet_li_bestreg.yaml"


## 2. Locked hashes and fair-control protocol

Validate the exact split, seed, optimizer, transformations, LI semantics, BESTREG rule, and the intended architecture-only delta against P1-noSSL.


In [2]:
fairness = verify_p2_fairness()
context = load_experiment_context(CONFIG_PATH)
assert context.split_sha256 == "8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f"
assert context.config["timm_model_name"] == "mobilevitv2_050.cvnets_in1k"
assert context.config["initialization"] == "imagenet" and context.config["ssl_checkpoint"] is None
assert context.config["use_li"] is True
assert context.config["selection_formula"] == "(SM0_RMSE + SM20_RMSE) / 2"
print_environment(context)
print(fairness)


python: 3.11.15
torch: 2.6.0+cu118
torchvision: 0.21.0+cu118
torchaudio: 2.6.0+cu118
timm: 1.0.29
cuda_available: True
cuda_device_count: 1
gpu: NVIDIA GeForce RTX 3050
torch_cuda_runtime: 11.8
seed: 20260905
config_sha256: b322ffea95daafb372f87404f1baf8ca8c80b33ffea011d0b3b5986c0d5e34d0
manifest_sha256: 8ff45054d4b8e3df9758d0c112dc16719572b2267906fb3a5ed5b3262a6732bd
split_sha256: 8927b8223b8c4c234d264ad9ea62ac2df6161124a79787a2e71eeb5cd23eae2f
{'status': 'PASS', 'matched_fields': ['dataset_manifest', 'dataset_manifest_sha256', 'split', 'split_sha256', 'split_counts', 'seed', 'epochs', 'batch_size', 'optimizer', 'learning_rate', 'weight_decay', 'loss', 'metrics', 'image_size', 'normalization_mean', 'normalization_std', 'train_augmentation', 'validation_transform', 'use_amp', 'num_workers', 'use_li', 'num_classes', 'class_mapping', 'primary_task', 'save_validation_best', 'checkpoint_selection', 'selection_formula', 'primary_checkpoint', 'epoch_60_checkpoint', 'test_evaluation'], 'inten

## 3. CUDA gate and new-output safety

The external P2 destination must be absent or contain only a valid rolling resume. A completed run is never overwritten.


In [3]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU_BLOCKED: P2 training requires CUDA")
metadata_path = context.run_dir / "run_metadata.json"
if metadata_path.is_file() and json.loads(metadata_path.read_text(encoding="utf-8")).get("training_completed") is True:
    raise RuntimeError("STOP: P2 output directory already contains a completed run")
print({"device": torch.cuda.get_device_name(0), "output_directory": str(context.run_dir), "test": "NOT_OPENED"})


{'device': 'NVIDIA GeForce RTX 3050', 'output_directory': '/mnt/d/check point/soilnet_final_runs/P2_MOBILEVITV2_IMAGENET_LI_BESTREG', 'test': 'NOT_OPENED'}


## 4. Temporary-model preflight

Build a disposable ImageNet-initialized baseline and check one train/validation batch. No optimizer step, checkpoint, or research metric is produced.


In [4]:
preflight = run_one_batch_preflight(context, device=torch.device("cuda"))
if preflight.get("status") != "PASS" or preflight.get("optimizer_step_performed") is not False or preflight.get("test_loader_instantiated") is not False:
    raise RuntimeError("P2_PREFLIGHT_FAILED")
print(preflight)


model.safetensors: reconstructing file:   0%|          |  0.00B / 5.54MB            

model.safetensors: downloading bytes:           |  0.00B            

{'status': 'PASS', 'device': 'cuda', 'train_batches': 1, 'validation_batches': 1, 'backward_batches': 1, 'train_samples': 1407, 'validation_samples': 289, 'train_image_shape': [32, 3, 224, 224], 'train_li_shape': [32, 1], 'train_regression_target_shape': [32, 2], 'train_class_target_shape': [32], 'output_shapes': [[32, 2], [32, 10]], 'load_report': None, 'loss_components_finite': {'total_loss': True, 'regression_loss': True, 'classification_loss': True}, 'optimizer': 'Adam', 'optimizer_step_performed': False, 'temporary_model': True, 'research_metrics_created': False, 'test_loader_instantiated': False, 'li_consumed_by_model': True, 'vicreg_checkpoint_loaded': False, 'initialization_provenance': {'source': 'timm/mobilevitv2_050.cvnets_in1k', 'timm_model_name': 'mobilevitv2_050.cvnets_in1k', 'pretrained_tag': 'cvnets_in1k', 'pretrained_family': 'ImageNet', 'vicreg_checkpoint_loaded': False}}


## 5. Automatic 60-epoch supervised run

Train all 60 epochs with Adam 1e-4, batch 32, weight decay 0, and no early stopping. Strictly lower mean validation regression RMSE replaces the best checkpoint.


In [5]:
run_metadata = train_experiment(context, resume_if_available=True)
assert run_metadata["training_completed"] is True
assert run_metadata["epochs_completed"] == 60
assert run_metadata["test_evaluated"] == "NO"


BEST REGRESSION UPDATED | epoch=1 | mean_RMSE=44.1081 | SM0=43.0362 | SM20=45.1799
{"epoch": 1, "train_total_loss": 2.5966603322462602, "train_regression_loss": 0.3120945692062378, "train_classification_loss": 2.2845657576214182, "validation_total_loss": 2.5022631168365477, "SM0_RMSE": 43.03623057857935, "SM0_MAE": 34.82331081894855, "SM20_RMSE": 45.17994986400586, "SM20_MAE": 37.415468486664, "classification_accuracy": 0.12802768166089964, "Macro-F1": 0.10626029247399782, "regression_metric_scale": "original_0_to_100_percentage_points"}
BEST REGRESSION UPDATED | epoch=2 | mean_RMSE=26.4057 | SM0=26.9383 | SM20=25.8732
{"epoch": 2, "train_total_loss": 2.390812039375305, "train_regression_loss": 0.16315320870754393, "train_classification_loss": 2.2276588299057702, "validation_total_loss": 2.273251461982727, "SM0_RMSE": 26.938259495083106, "SM0_MAE": 20.978575490750245, "SM20_RMSE": 25.873206230961493, "SM20_MAE": 19.980105034310924, "classification_accuracy": 0.17993079584775087, "Macro

## 6. Freeze all four models

Verify P0, both P1 ablations, and P2 checkpoint identities plus validation artifacts, then write the registry while the test firewall remains closed.


In [6]:
registry = write_frozen_model_registry()
assert registry["model_count"] == 4
assert registry["TEST_OPENED"] == "NO" and registry["MODEL_SET_FROZEN"] == "YES"
print(json.dumps({"registry": "results/model_registry/frozen_model_registry.json", "models": [m["experiment_id"] for m in registry["models"]], "TEST_OPENED": "NO"}, indent=2))


{
  "registry": "results/model_registry/frozen_model_registry.json",
  "models": [
    "P0_FINAL_SOILNET_VICREG_MU27_LI_V4_BESTREG",
    "P1_SOILNET_VICREG_MU27_NO_LI_BESTREG",
    "P1_SOILNET_IMAGENET_LI_NO_SSL_BESTREG",
    "P2_MOBILEVITV2_IMAGENET_LI_BESTREG"
  ],
  "TEST_OPENED": "NO"
}
